In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 130
plt.rcParams['savefig.dpi'] = 180

def encontrar_pasta_2026(inicio=Path.cwd()):
    inicio = inicio.resolve()
    for pasta in (inicio, *inicio.parents):
        candidata = pasta / '2026'
        if candidata.exists():
            return candidata
    raise FileNotFoundError('Nao foi encontrada a pasta 2026 a partir do diretorio atual.')

PASTA_2026 = encontrar_pasta_2026()
PASTA_SONDA = PASTA_2026 / 'Dados SONDA'
PASTA_SONDA_TRATADA = PASTA_2026 / 'dados_tratados_sonda'
PASTA_SAIDA = PASTA_2026 / 'visualizacoes_comparativas_bases'
PASTA_DADOS = PASTA_SAIDA / 'dados_padronizados'
PASTA_FONTE = PASTA_SAIDA / 'figuras_por_fonte'
PASTA_COMPARACAO = PASTA_SAIDA / 'comparacao_entre_fontes'
PASTA_TABELAS = PASTA_SAIDA / 'tabelas'
for pasta in (PASTA_DADOS, PASTA_FONTE, PASTA_COMPARACAO, PASTA_TABELAS):
    pasta.mkdir(parents=True, exist_ok=True)

# Ajuste somente se a verificacao documental indicar diferenca de fuso entre as fontes.
DESLOCAMENTOS_HORAS = {'sonda_md': 0}
INVALIDOS = [-9999, -999, -6999, 7999, 9999]
FREQUENCIA_VISUAL = 'D'  # media diaria para figuras legiveis em series longas
INICIO_ANALISE = pd.Timestamp('2024-01-05 00:00:00')
FIM_ANALISE = pd.Timestamp('2026-01-06 00:00:00')  # limite exclusivo; acompanha a NASA POWER

print(f'Pasta de saida: {PASTA_SAIDA}')

In [ ]:
def converter_numericas(df, colunas):
    for coluna in colunas:
        if coluna in df.columns:
            df[coluna] = pd.to_numeric(df[coluna], errors='coerce').replace(INVALIDOS, np.nan)
    return df


def ler_sonda_md():
    arquivos = sorted(PASTA_SONDA.rglob('*_MD.dat'))
    if not arquivos:
        raise FileNotFoundError('Arquivos brutos SONDA-MD (*_MD.dat) nao encontrados em Dados SONDA.')

    partes = []
    for arquivo in arquivos:
        bruto = pd.read_csv(
            arquivo, skiprows=[0, 2, 3],
            usecols=lambda coluna: coluna in ['TIMESTAMP', 'tp_sfc', 'humid'],
            low_memory=False,
        )
        bruto['TIMESTAMP'] = pd.to_datetime(bruto['TIMESTAMP'], errors='coerce')
        partes.append(converter_numericas(bruto, ['tp_sfc', 'humid']).dropna(subset=['TIMESTAMP']))

    dados = pd.concat(partes, ignore_index=True).rename(columns={'TIMESTAMP': 'data_hora'})
    dados = dados.sort_values('data_hora').drop_duplicates('data_hora')
    dados['data_hora'] += pd.Timedelta(hours=DESLOCAMENTOS_HORAS['sonda_md'])
    dados['hora'] = dados['data_hora'].dt.floor('h')
    dados_horarios = (
        dados.groupby('hora', as_index=False)[['tp_sfc', 'humid']].mean()
        .rename(columns={'hora': 'data_hora', 'tp_sfc': 'temperatura', 'humid': 'umidade_relativa'})
    )
    return dados_horarios.loc[
        (dados_horarios['data_hora'] >= INICIO_ANALISE)
        & (dados_horarios['data_hora'] < FIM_ANALISE)
    ].reset_index(drop=True)

sonda_md = ler_sonda_md()
arquivo_sonda_md = PASTA_SONDA_TRATADA / 'sonda_md_horario_padronizado.csv'
sonda_md.to_csv(arquivo_sonda_md, index=False)
print(
    f'SONDA-MD: {len(sonda_md)} linhas | '
    f'{sonda_md.data_hora.min()} a {sonda_md.data_hora.max()} | '
    f'salvo em {arquivo_sonda_md}'
)

In [ ]:
ROTULOS = {
    'radiacao_onda_curta': 'Radiacao de onda curta (W/m2)',
    'glo_avg': 'Radiacao global (W/m2)',
    'dif_avg': 'Radiacao difusa (W/m2)',
    'dir_avg': 'Radiacao direta (W/m2)',
    'lw_raw_avg': 'Radiacao de onda longa bruta (W/m2)',
    'lw_calc_avg': 'Radiacao de onda longa calculada (W/m2)',
    'par_avg': 'PAR (umol/s m2)',
    'lux_avg': 'Iluminancia (klux)',
    'tilt_avg': 'Radiacao em plano inclinado (W/m2)',
    'temperatura': 'Temperatura (C)',
    'umidade_relativa': 'Umidade relativa (%)',
    'pressao': 'Pressao (mb)',
    'chuva': 'Precipitacao (mm/h)',
    'velocidade_vento': 'Velocidade do vento (m/s)',
    'direcao_vento': 'Direcao do vento (graus)',
}

def serie_visual(dados, coluna):
    return dados.set_index('data_hora')[coluna].resample(FREQUENCIA_VISUAL).mean()

def plotar_evolucao(dados, nome, colunas, arquivo, ncol=1):
    colunas = [coluna for coluna in colunas if coluna in dados.columns]
    nlin = int(np.ceil(len(colunas) / ncol))
    fig, eixos = plt.subplots(nlin, ncol, figsize=(14, 3.6 * nlin), sharex=True, squeeze=False)
    for eixo, coluna in zip(eixos.flat, colunas):
        serie_visual(dados, coluna).plot(ax=eixo, color='#1f5a85', linewidth=0.8)
        eixo.set_title(ROTULOS.get(coluna, coluna))
        eixo.set_xlabel('')
        eixo.set_ylabel('')
    for eixo in eixos.flat[len(colunas):]:
        eixo.remove()
    inicio = dados['data_hora'].min().strftime('%Y')
    fim = dados['data_hora'].max().strftime('%Y')
    fig.suptitle(f'Evolucao temporal - {nome} ({inicio}–{fim}; {FREQUENCIA_VISUAL})', y=1.01)
    fig.tight_layout()
    fig.savefig(arquivo, bbox_inches='tight')
    plt.show()
    plt.close(fig)

plotar_evolucao(sonda_md, 'SONDA-MD', ['temperatura', 'umidade_relativa'], PASTA_FONTE / 'sonda_md_evolucao.png', ncol=2)